# PowerOps_v1 — Notebook 03: Pinecone Ingestion

This notebook embeds the prepared Documents from Notebook 02
(`data/prepared_documents.json`) and upserts them into a Pinecone index.

It is the first notebook that talks to external paid services (OpenAI
embeddings, Pinecone), so all configuration comes from `.env` — **never**
hardcoded here.

### Idempotency

Each vector is upserted with the stable ID we built in Notebook 02
(`{issue_key}::chunk{chunk_id}`). Pinecone's `upsert` is an overwrite-by-ID
operation, so re-running this notebook updates existing vectors in place
instead of creating duplicates.


## 1. Load configuration from `.env`

In [1]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("../.env"))

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
PINECONE_INDEX_NAME = os.environ.get("PINECONE_INDEX_NAME", "powerops-v1")
PINECONE_CLOUD = os.environ.get("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.environ.get("PINECONE_REGION", "us-east-1")

EMBEDDING_PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "openai")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.environ.get("EMBEDDING_DIMENSIONS", "1536"))

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Never print secret values — only confirm presence and length.
print(f"PINECONE_INDEX_NAME:   {PINECONE_INDEX_NAME}")
print(f"PINECONE_CLOUD/REGION: {PINECONE_CLOUD}/{PINECONE_REGION}")
print(f"EMBEDDING_PROVIDER:    {EMBEDDING_PROVIDER}")
print(f"EMBEDDING_MODEL:       {EMBEDDING_MODEL}")
print(f"EMBEDDING_DIMENSIONS:  {EMBEDDING_DIMENSIONS}")
print(f"PINECONE_API_KEY set:  {bool(PINECONE_API_KEY)} (length {len(PINECONE_API_KEY)})")
print(f"OPENAI_API_KEY set:    {bool(OPENAI_API_KEY)} (length {len(OPENAI_API_KEY)})")


PINECONE_INDEX_NAME:   powerops-v1
PINECONE_CLOUD/REGION: aws/us-east-1
EMBEDDING_PROVIDER:    openai
EMBEDDING_MODEL:       text-embedding-3-small
EMBEDDING_DIMENSIONS:  1536
PINECONE_API_KEY set:  True (length 75)
OPENAI_API_KEY set:    True (length 164)


## 2. Load prepared documents from Notebook 02

In [2]:
import json

from langchain_core.documents import Document

PREPARED_PATH = Path("../data/prepared_documents.json")

with open(PREPARED_PATH, "r", encoding="utf-8") as f:
    prepared = json.load(f)

documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in prepared]
vector_ids = [item["vector_id"] for item in prepared]

print(f"Loaded {len(documents)} prepared documents (and {len(vector_ids)} vector IDs).")
print(documents[0].page_content)
print(vector_ids[0])


Loaded 1000 prepared documents (and 1000 vector IDs).
Issue Key: INO-21920
Summary: SDX case  are creating but remaining in Delayed Processing Pending status -FT13
Assigned Team: Falcon Squad
Assignee: Sullivan, Deepa (Contractor)
Reporter: Sullivan, Deepa (Contractor)
Priority: Medium
Status: In Progress
Story Points: 0.25
Last Updated: 5/18/26 12:01
Due Date: 5/15/26 0:00
INO-21920::chunk0


## 3. Initialize the embedding model (configurable)

Only `EMBEDDING_PROVIDER=openai` is implemented for this POC, matching the
project's current configuration. The branch is structured so a future
provider (e.g. a different `langchain_*` embeddings class) can be added
without touching any other notebook or module — everything downstream only
depends on the `embeddings` object's `.embed_query()` / `.embed_documents()`
interface, not on which provider produced it.


In [3]:
def build_embeddings():
    """Build the embedding client based on EMBEDDING_PROVIDER."""
    if EMBEDDING_PROVIDER == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY)
    raise ValueError(f"Unsupported EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER}")


embeddings = build_embeddings()
print(f"Initialized embeddings client: {type(embeddings).__name__} (model={EMBEDDING_MODEL})")


Initialized embeddings client: OpenAIEmbeddings (model=text-embedding-3-small)


## 4. Verify embedding dimensions

Before creating (or connecting to) a Pinecone index, confirm the embedding
model's actual output dimension matches `EMBEDDING_DIMENSIONS` from `.env`.
A mismatch here would cause every upsert to fail later with a much less
clear error, so we fail fast with a readable message instead.


In [4]:
probe_vector = embeddings.embed_query("PowerOps dimension probe")
actual_dim = len(probe_vector)

print(f"Configured EMBEDDING_DIMENSIONS: {EMBEDDING_DIMENSIONS}")
print(f"Actual embedding dimension:      {actual_dim}")

if actual_dim != EMBEDDING_DIMENSIONS:
    raise ValueError(
        f"Embedding dimension mismatch: .env says {EMBEDDING_DIMENSIONS}, "
        f"but '{EMBEDDING_MODEL}' actually returns {actual_dim}-dim vectors. "
        f"Update EMBEDDING_DIMENSIONS in .env to {actual_dim}."
    )

print("Dimension check passed.")


Configured EMBEDDING_DIMENSIONS: 1536
Actual embedding dimension:      1536
Dimension check passed.


## 5. Initialize Pinecone and create/connect to the index

`create_index` is only called if an index with this name doesn't already
exist — safe to re-run.


In [5]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = {idx["name"] for idx in pc.list_indexes()}
print(f"Existing Pinecone indexes: {existing_indexes}")

if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"Creating index '{PINECONE_INDEX_NAME}' ({actual_dim}-dim, cosine, "
          f"serverless {PINECONE_CLOUD}/{PINECONE_REGION})...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=actual_dim,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    # Wait until the index reports ready before using it.
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
    print("Index created and ready.")
else:
    print(f"Index '{PINECONE_INDEX_NAME}' already exists — connecting to it.")

index = pc.Index(PINECONE_INDEX_NAME)
index.describe_index_stats()


Existing Pinecone indexes: set()
Creating index 'powerops-v1' (1536-dim, cosine, serverless aws/us-east-1)...


Index created and ready.


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

## 6. Connect LangChain's Pinecone vector store wrapper

This is what Notebooks 04–05 will use for filtered similarity search.


In [6]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)
print("PineconeVectorStore ready.")


PineconeVectorStore ready.


## 7. Idempotent upsert

We upsert in batches using our own precomputed `vector_ids` (not
auto-generated IDs) so re-running this notebook **updates** existing vectors
rather than duplicating them. We batch to stay well under Pinecone's
per-request payload limits and to avoid a single giant embeddings call.


In [7]:
BATCH_SIZE = 100

def batched(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


total_upserted = 0
for doc_batch, id_batch in zip(batched(documents, BATCH_SIZE), batched(vector_ids, BATCH_SIZE)):
    vector_store.add_documents(documents=doc_batch, ids=id_batch)
    total_upserted += len(doc_batch)
    print(f"Upserted {total_upserted}/{len(documents)}...")

print(f"Done. Upserted {total_upserted} vectors.")


Upserted 100/1000...


Upserted 200/1000...


Upserted 300/1000...


Upserted 400/1000...


Upserted 500/1000...


Upserted 600/1000...


Upserted 700/1000...


Upserted 800/1000...


Upserted 900/1000...


Upserted 1000/1000...
Done. Upserted 1000 vectors.


## 8. Verify vector count

Pinecone's index stats can lag a few seconds behind the most recent upsert
(eventual consistency), so we poll briefly if the count looks stale.


In [8]:
expected_count = len(documents)

for attempt in range(10):
    stats = index.describe_index_stats()
    current_count = stats["total_vector_count"]
    if current_count >= expected_count:
        break
    time.sleep(2)

print(f"Expected vectors: {expected_count}")
print(f"Reported vectors: {current_count}")
if current_count < expected_count:
    print("Note: Pinecone stats can take a short while to fully reflect a large "
          "upsert — re-run this cell if the count still looks low.")


Expected vectors: 1000
Reported vectors: 1000


## 9. Test similarity searches

A handful of purely semantic queries (no metadata filters yet — that's
Notebook 04) to sanity-check that embeddings and retrieval are actually
working end-to-end.


In [9]:
TEST_QUERIES = [
    "Kubernetes production problems",
    "certificate failures",
    "pipeline deployment errors",
    "database availability problems",
]

def print_results(query, results):
    print("=" * 70)
    print(f"QUERY: {query}")
    print("-" * 70)
    if not results:
        print("  (no results)")
        return
    for doc, score in results:
        m = doc.metadata
        print(f"  [{score:.4f}] {m.get('issue_key')} | {m.get('assigned_team')} | "
              f"{m.get('assignee')} | priority={m.get('priority')} | status={m.get('status')}")
        # Trim to just the Summary line for a quick read
        summary_line = next((l for l in doc.page_content.splitlines() if l.startswith("Summary:")), "")
        print(f"      {summary_line}")


for query in TEST_QUERIES:
    results = vector_store.similarity_search_with_score(query, k=5)
    print_results(query, results)


QUERY: Kubernetes production problems
----------------------------------------------------------------------
  [0.4361] INO-21851 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
      Summary: Dynatrace Prod Alert - Multiple service problems - P-26051174
  [0.4353] INO-20176 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
      Summary: Dynatrace Prod Alert - Multiple application problems - P-2604936
  [0.4159] INO-20237 | Nova Team | Gibson, Anjali | priority=Medium | status=Soft Delete
      Summary: Dynatrace Prod Alert - Failure rate increase - P-26041785
  [0.4147] INO-20235 | Nova Team | Gibson, Anjali | priority=Medium | status=Soft Delete
      Summary: Dynatrace Prod Alert - Failure rate increase - P-26041783
  [0.4146] INO-20241 | Nova Team | Gibson, Anjali | priority=Medium | status=Soft Delete
      Summary: Dynatrace Prod Alert - Failure rate increase - P-26041780


QUERY: certificate failures
----------------------------------------------------------------------
  [0.4209] INO-19926 | Nova Team | Myers, Deepa | priority=Medium | status=Done
      Summary: Getting IBM Cert error on PCP101 env
  [0.4076] INO-20582 | Falcon Squad | Bose, Alexander | priority=Medium | status=To Do
      Summary: Cert IDPPNCID2 - Certificate for NCID site in Pre-Prod  Expires: 03/27/2027
  [0.3937] INO-20522 | Falcon Squad | Mason, Anthony | priority=Medium | status=Rejected
      Summary: Webservices PPE Certificate renewal
  [0.3850] INO-19900 | Nova Team | Shaw, Elizabeth | priority=Medium | status=Done
      Summary: Edward Wallace_CC_S_CCFRCRECLASS_RPT_PKG is failing in FT06
  [0.3827] INO-20771 | Nova Team | Gibson, Anjali | priority=Medium | status=To Do
      Summary: Update and Validate Partner Certificates withing James Narayan ENV


QUERY: pipeline deployment errors
----------------------------------------------------------------------
  [0.4602] INO-20433 | Nova Team | Porter, Anjali (Contractor) | priority=Medium | status=Done
      Summary: Investigate reporting deployment failed job
  [0.4584] INO-21285 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
      Summary: Scheduled Manual(Curam)-FT1/FT2/PCP6-Hotfix-4/30/2026-EB26Q2R2.0.1-#B1 - CodeDeploy
  [0.4575] INO-20627 | Falcon Squad | Mason, Ashley (Contractor) | priority=Medium | status=Done
      Summary: Scheduled Manual(Curam)-FT1/FT2/PCP6-Hotfix-4/17/2026-EB26Q2R1.0.5-#B1 - CodeDeploy
  [0.4572] INO-21287 | Summit Crew | Rice, Anjali (Contractor) | priority=Medium | status=Done
      Summary: Scheduled Manual(Curam)-gporter Gomez-Hotfix-5/3/2026-EB26Q2R2.0.1-#B1 - CodeDeploy
  [0.4545] INO-21720 | Nova Team | Banerjee, Ashley (Contractor) | priority=Medium | status=In Progress
      Summary: Incremental deploy in FT22


QUERY: database availability problems
----------------------------------------------------------------------
  [0.4099] INO-20557 | Falcon Squad | Bose, Jacob (Contractor) | priority=Medium | status=Done
      Summary: Request to have two DEV databases for curam building two branches on VM
  [0.4024] INO-21218 | Nova Team | Shaw, Elizabeth | priority=Medium | status=Done
      Summary: Database read access for mentioned environments
  [0.4001] INO-21348 | Nova Team | Gibson, Anjali | priority=Medium | status=Done
      Summary: Dynatrace Prod Alert - Failed database connects - P-26058
  [0.3960] INO-21628 | Nova Team | Bose, Jacob (Contractor) | priority=Medium | status=Done
      Summary: Provide local database access for user - nbhaskar
  [0.3932] INO-20782 | Nova Team | Myers, Deepa | priority=Medium | status=Done
      Summary: In Ftest19 getting document management system is unavailable


## Summary & next steps

- Loaded `.env` configuration — no secrets hardcoded, none printed.
- Verified the OpenAI embedding model's actual output dimension matches
  configuration before touching Pinecone.
- Created (or connected to) the `powerops-v1` serverless Pinecone index.
- Upserted all prepared documents using stable, precomputed vector IDs
  (`{issue_key}::chunk{chunk_id}`) — **idempotent**: re-running this notebook
  updates the same vectors rather than duplicating them.
- Verified the resulting vector count matches the number of prepared
  documents.
- Ran test semantic queries and confirmed relevant issues come back with
  their `issue_key`, team, assignee, priority, and status.

**Next: Notebook 04 — Metadata-Aware Retrieval.** We'll build a deterministic
query parser (using the real vocabulary from Notebook 01's
`data/vocabulary.json`) that detects structured filters in a question, and
combine them with semantic search via Pinecone's metadata filtering — instead
of relying on semantic similarity alone for questions that clearly specify a
team, assignee, priority, or status.
